[Source](https://github.com/antimatter15/splat/blob/main/convert.py)

In [1]:
!pip install plyfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
!cp '/content/drive/MyDrive/Sequim Shipwreck/sequim-shipwreck_brush.ply' /content/sequim-shipwreck.ply

In [5]:
# You can use this to convert a .ply file to a .splat file programmatically in python
# Alternatively you can drag and drop a .ply file into the viewer at https://antimatter15.com/splat

from plyfile import PlyData
import numpy as np
import argparse
from io import BytesIO


def process_ply_to_splat(ply_file_path):
    plydata = PlyData.read(ply_file_path)
    vert = plydata["vertex"]
    sorted_indices = np.argsort(
        -np.exp(vert["scale_0"] + vert["scale_1"] + vert["scale_2"])
        / (1 + np.exp(-vert["opacity"]))
    )
    buffer = BytesIO()
    for idx in sorted_indices:
        v = plydata["vertex"][idx]
        position = np.array([v["x"], v["y"], v["z"]], dtype=np.float32)
        scales = np.exp(
            np.array(
                [v["scale_0"], v["scale_1"], v["scale_2"]],
                dtype=np.float32,
            )
        )
        rot = np.array(
            [v["rot_0"], v["rot_1"], v["rot_2"], v["rot_3"]],
            dtype=np.float32,
        )
        SH_C0 = 0.28209479177387814
        color = np.array(
            [
                0.5 + SH_C0 * v["f_dc_0"],
                0.5 + SH_C0 * v["f_dc_1"],
                0.5 + SH_C0 * v["f_dc_2"],
                1 / (1 + np.exp(-v["opacity"])),
            ]
        )
        buffer.write(position.tobytes())
        buffer.write(scales.tobytes())
        buffer.write((color * 255).clip(0, 255).astype(np.uint8).tobytes())
        buffer.write(
            ((rot / np.linalg.norm(rot)) * 128 + 128)
            .clip(0, 255)
            .astype(np.uint8)
            .tobytes()
        )

    return buffer.getvalue()


def save_splat_file(splat_data, output_path):
    with open(output_path, "wb") as f:
        f.write(splat_data)

In [6]:
input_file = 'sequim-shipwreck.ply'
output_file = input_file.split('.')[0] + '.splat'

splat_data = process_ply_to_splat(input_file)

save_splat_file(splat_data, output_file)

ValueError: no field of name scale_0

In [7]:
from plyfile import PlyData

plydata = PlyData.read("sequim-shipwreck.ply")
print(plydata['vertex'].data.dtype.names)

('x', 'y', 'z', 'red', 'green', 'blue')


In [8]:
from plyfile import PlyData
import numpy as np
from io import BytesIO

def process_simple_ply_to_splat(ply_file_path):
    plydata = PlyData.read(ply_file_path)
    vert = plydata["vertex"]

    buffer = BytesIO()
    for v in vert:
        position = np.array([v["x"], v["y"], v["z"]], dtype=np.float32)
        # Use fixed scale
        scales = np.array([1.0, 1.0, 1.0], dtype=np.float32)
        # Color
        color = np.array([v["red"], v["green"], v["blue"], 255], dtype=np.uint8)
        # Rotation (identity)
        rot = np.array([128, 128, 128, 128], dtype=np.uint8)

        buffer.write(position.tobytes())
        buffer.write(scales.tobytes())
        buffer.write(color.tobytes())
        buffer.write(rot.tobytes())

    return buffer.getvalue()

def save_splat_file(splat_data, output_path):
    with open(output_path, "wb") as f:
        f.write(splat_data)


input_file = 'sequim-shipwreck.ply'
output_file = input_file.split('.')[0] + '.splat'

splat_data = process_simple_ply_to_splat(input_file)
save_splat_file(splat_data, output_file)


In [10]:
!cp sequim-shipwreck.splat '/content/drive/MyDrive/Sequim Shipwreck/sequim-shipwreck_brush.splat'